## Install Dependencies

In [ ]:
!pip install -q transformers sentencepiece accelerate torch bert-score rouge-score matplotlib seaborn pandas numpy

## Load Three LLMs

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

MODELS = {
    "flan-t5-base": "google/flan-t5-base",
    "flan-t5-large": "google/flan-t5-large",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

loaded_models = {}

for alias, model_id in MODELS.items():
    print(f"Loading {alias} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if "t5" in model_id.lower():
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id, torch_dtype=torch.float32).to(device)
        model_type = "seq2seq"
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
        model_type = "causal"
    model.eval()
    loaded_models[alias] = {"tokenizer": tokenizer, "model": model, "type": model_type}
    print(f"  {alias} loaded")

print("All models loaded")

## Generate Function (Per Model)

In [ ]:
def generate_with(alias: str, prompt: str, max_new_tokens: int = 120) -> tuple[str, float]:
    entry = loaded_models[alias]
    tokenizer = entry["tokenizer"]
    model = entry["model"]
    model_type = entry["type"]

    if model_type == "causal":
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512).to(device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id if tokenizer.pad_token_id is None else tokenizer.pad_token_id,
        )
    elapsed = time.time() - start

    if model_type == "causal":
        input_len = inputs["input_ids"].shape[1]
        new_tokens = outputs[0][input_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    else:
        text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    return text, elapsed

print("Generate function ready")

## MessageBus

In [ ]:
from datetime import datetime

class MessageBus:
    def __init__(self):
        self.messages: list[dict] = []

    def post(self, sender: str, content: str, model: str = "", latency: float = 0.0) -> None:
        self.messages.append({
            "sender": sender,
            "content": content,
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "model": model,
            "latency": latency,
        })

    def get_recent(self, n: int = 6) -> list[dict]:
        return self.messages[-n:]

    def format_context(self, n: int = 6) -> str:
        history = self.get_recent(n)
        return "\n".join(f"{m['sender']}: {m['content']}" for m in history)

    def summary(self) -> str:
        senders = set(m["sender"] for m in self.messages)
        return f"Total messages: {len(self.messages)} | Participants: {', '.join(senders)}"

    def print_transcript(self) -> None:
        print("=" * 70)
        print("FULL CONVERSATION TRANSCRIPT")
        print("=" * 70)
        for m in self.messages:
            model_tag = f" [{m['model']}]" if m['model'] else ""
            latency_tag = f" ({m['latency']:.2f}s)" if m['latency'] else ""
            print(f"[{m['timestamp']}] {m['sender']:25s}{model_tag}{latency_tag}:")
            print(f"  {m['content']}")
            print()

print("MessageBus ready")

## Agent Class (Multi-LLM Aware)

In [ ]:
class Agent:
    def __init__(self, name: str, role: str, bus: MessageBus, model_alias: str,
                 expertise: list[str] | None = None, max_new_tokens: int = 120):
        self.name = name
        self.role = role
        self.bus = bus
        self.model_alias = model_alias
        self.expertise = expertise or []
        self.max_new_tokens = max_new_tokens
        self.metrics: list[dict] = []

    def _build_prompt(self) -> str:
        context = self.bus.format_context(n=6)
        expertise = ", ".join(self.expertise) if self.expertise else self.role
        prompt = (
            f"You are {self.name}, an expert in {expertise}.\n"
            f"Read this conversation and give a short, practical 2-sentence response:\n\n"
            f"{context}\n\n"
            f"{self.name}'s response:"
        )
        return prompt

    def think_and_act(self) -> str:
        prompt = self._build_prompt()
        response, latency = generate_with(self.model_alias, prompt, self.max_new_tokens)

        if response.lower().startswith(self.name.lower()):
            response = response[len(self.name):].lstrip(":").strip()

        word_count = len(response.split())
        self.metrics.append({"latency": latency, "word_count": word_count, "model": self.model_alias})
        self.bus.post(self.name, response, model=self.model_alias, latency=latency)
        return response

    def __repr__(self):
        return f"Agent(name={self.name!r}, model={self.model_alias!r})"

print("Agent class ready")

## Orchestrator

In [ ]:
def run_a2a(task: str, agents: list, bus: MessageBus, rounds: int = 2, verbose: bool = True) -> None:
    bus.post("User", task)
    sep = "=" * 70

    if verbose:
        print(sep)
        print(f"TASK: {task}")
        print(sep)

    for r in range(1, rounds + 1):
        if verbose:
            print(f"\n{'─'*70}")
            print(f"  ROUND {r} / {rounds}")
            print(f"{'─'*70}")

        for agent in agents:
            response = agent.think_and_act()
            if verbose:
                print(f"\n[{agent.name}] (model: {agent.model_alias})")
                print(f"  {response}")

    if verbose:
        print(f"\n{sep}")
        print(f"Done | {bus.summary()}")
        print(sep)

print("Orchestrator ready")

## Create Agents (One Per LLM) and Run

In [ ]:
bus = MessageBus()

agents = [
    Agent(
        name="Climate Scientist",
        role="climate scientist",
        bus=bus,
        model_alias="flan-t5-base",
        expertise=["carbon budgets", "IPCC data", "methane emissions", "tipping points"],
    ),
    Agent(
        name="Policy Maker",
        role="environmental policy maker",
        bus=bus,
        model_alias="flan-t5-large",
        expertise=["Paris Agreement", "carbon pricing", "green subsidies", "international law"],
    ),
    Agent(
        name="Green Tech Engineer",
        role="renewable energy engineer",
        bus=bus,
        model_alias="TinyLlama",
        expertise=["solar power", "wind energy", "battery storage", "hydrogen fuel cells"],
    ),
]

run_a2a(
    task="What is the single most impactful action to reduce global carbon emissions?",
    agents=agents,
    bus=bus,
    rounds=2,
    verbose=True,
)

## Full Transcript

In [ ]:
bus.print_transcript()

## Evaluation — BERTScore + ROUGE + Word Count

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd
import numpy as np

task_text = "What is the single most impactful action to reduce global carbon emissions?"

agent_messages = [m for m in bus.messages if m["sender"] != "User"]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

eval_rows = []

for m in agent_messages:
    rouge = scorer.score(task_text, m["content"])
    row = {
        "Agent": m["sender"],
        "Model": m["model"],
        "Word Count": len(m["content"].split()),
        "Latency (s)": round(m["latency"], 2),
        "ROUGE-1": round(rouge["rouge1"].fmeasure, 3),
        "ROUGE-L": round(rouge["rougeL"].fmeasure, 3),
    }
    eval_rows.append(row)

responses = [m["content"] for m in agent_messages]
references = [task_text] * len(responses)
P, R, F1 = bert_score(responses, references, lang="en", verbose=False)

for i, row in enumerate(eval_rows):
    row["BERTScore F1"] = round(F1[i].item(), 3)

eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))

## Visual 1 — BERTScore F1 by Agent and Model

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

model_colors = {
    "flan-t5-base": "#4C9BE8",
    "flan-t5-large": "#F4A03A",
    "TinyLlama": "#5CB85C",
}

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{r['Agent']}\n({r['Model']})" for r in eval_rows]
scores = [r["BERTScore F1"] for r in eval_rows]
colors = [model_colors.get(r["Model"], "#999") for r in eval_rows]

bars = ax.bar(labels, scores, color=colors, edgecolor="white", linewidth=1.2, width=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{score:.3f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("BERTScore F1 — Agent Responses vs Task", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("BERTScore F1")
ax.set_ylim(0, 1.0)
ax.set_xlabel("Agent (Model)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

patches = [mpatches.Patch(color=c, label=m) for m, c in model_colors.items()]
ax.legend(handles=patches, title="Model", loc="lower right")
plt.tight_layout()
plt.savefig("bertscore_comparison.png", dpi=150)
plt.show()

## Visual 2 — Latency per Agent per Round

In [ ]:
latency_data = {alias: [] for alias in MODELS.keys()}
for m in agent_messages:
    latency_data[m["model"]].append(m["latency"])

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, 3)
width = 0.25
offsets = [-width, 0, width]

for i, (alias, lats) in enumerate(latency_data.items()):
    ax.bar(x + offsets[i], lats[:2], width=width, label=alias,
           color=list(model_colors.values())[i], edgecolor="white")

ax.set_title("Inference Latency per Round by Model", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Round")
ax.set_ylabel("Latency (seconds)")
ax.set_xticks(x)
ax.set_xticklabels(["Round 1", "Round 2"])
ax.legend(title="Model")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("latency_comparison.png", dpi=150)
plt.show()

## Visual 3 — Word Count per Agent

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
wc_labels = [f"{r['Agent']}\n({r['Model']})" for r in eval_rows]
wc_values = [r["Word Count"] for r in eval_rows]
wc_colors = [model_colors.get(r["Model"], "#999") for r in eval_rows]

bars = ax.barh(wc_labels, wc_values, color=wc_colors, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, wc_values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=10, fontweight="bold")

ax.set_title("Response Word Count per Agent", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Word Count")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.4)
patches = [mpatches.Patch(color=c, label=m) for m, c in model_colors.items()]
ax.legend(handles=patches, title="Model", loc="lower right")
plt.tight_layout()
plt.savefig("wordcount_comparison.png", dpi=150)
plt.show()

## Visual 4 — ROUGE-1 vs ROUGE-L Comparison

In [ ]:
import seaborn as sns

rouge_df = eval_df[["Agent", "Model", "ROUGE-1", "ROUGE-L"]].melt(
    id_vars=["Agent", "Model"], var_name="Metric", value_name="Score"
)

fig, ax = plt.subplots(figsize=(11, 5))
x_labels = [f"{r['Agent']}\n({r['Model']})" for r in eval_rows]
x_pos = np.arange(len(x_labels))

r1_scores = eval_df["ROUGE-1"].tolist()
rl_scores = eval_df["ROUGE-L"].tolist()

ax.plot(x_labels, r1_scores, marker="o", linewidth=2, markersize=8, label="ROUGE-1", color="#4C9BE8")
ax.plot(x_labels, rl_scores, marker="s", linewidth=2, markersize=8, label="ROUGE-L", color="#F4A03A", linestyle="--")

for i, (r1, rl, lbl) in enumerate(zip(r1_scores, rl_scores, x_labels)):
    ax.annotate(f"{r1:.3f}", (lbl, r1), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    ax.annotate(f"{rl:.3f}", (lbl, rl), textcoords="offset points", xytext=(0, -14), ha="center", fontsize=9)

ax.set_title("ROUGE-1 vs ROUGE-L per Agent", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Score")
ax.set_ylim(0, 0.6)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("rouge_comparison.png", dpi=150)
plt.show()

## Visual 5 — Overall Model Scorecard (Radar Chart)

In [ ]:
from matplotlib.patches import FancyBboxPatch

model_summary = eval_df.groupby("Model").agg(
    BERTScore=("BERTScore F1", "mean"),
    ROUGE1=("ROUGE-1", "mean"),
    ROUGEL=("ROUGE-L", "mean"),
    AvgWords=("Word Count", "mean"),
    AvgLatency=("Latency (s)", "mean"),
).reset_index()

model_summary["SpeedScore"] = 1 - (model_summary["AvgLatency"] / model_summary["AvgLatency"].max())
model_summary["VerbosityScore"] = model_summary["AvgWords"] / model_summary["AvgWords"].max()

categories = ["BERTScore", "ROUGE-1", "ROUGE-L", "Speed", "Verbosity"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for _, row in model_summary.iterrows():
    values = [
        row["BERTScore"],
        row["ROUGE1"],
        row["ROUGEL"],
        row["SpeedScore"],
        row["VerbosityScore"],
    ]
    values += values[:1]
    color = model_colors.get(row["Model"], "#999")
    ax.plot(angles, values, linewidth=2, linestyle="solid", label=row["Model"], color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_title("Model Scorecard — Radar Comparison", size=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), title="Model")
plt.tight_layout()
plt.savefig("radar_scorecard.png", dpi=150)
plt.show()

## Try Your Own Question

In [ ]:
YOUR_QUESTION = "How should developing nations balance economic growth with climate commitments?"

fresh_bus = MessageBus()
for a in agents:
    a.bus = fresh_bus

run_a2a(
    task=YOUR_QUESTION,
    agents=agents,
    bus=fresh_bus,
    rounds=2,
    verbose=True,
)

## A2A Architecture — Multi-LLM Setup

```
┌─────────────────────────────────────────────────────────────────┐
│                         MessageBus                              │
│  [User Task] → [Scientist] → [Policy] → [Engineer] → ...        │
│                    READ ↕               WRITE ↕                  │
└─────────────────────────────────────────────────────────────────┘
         ↑                   ↑                    ↑
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│ Climate         │ │ Policy          │ │ Green Tech      │
│ Scientist       │ │ Maker           │ │ Engineer        │
│ flan-t5-base    │ │ flan-t5-large   │ │ TinyLlama-1.1B  │
│ ~250 MB         │ │ ~800 MB         │ │ ~1.1B params    │
│ READ bus        │ │ READ bus        │ │ READ bus        │
│ BUILD prompt    │ │ BUILD prompt    │ │ BUILD prompt    │
│ CALL LLM        │ │ CALL LLM        │ │ CALL LLM        │
│ POST reply      │ │ POST reply      │ │ POST reply      │
└────────┬────────┘ └────────┬────────┘ └────────┬────────┘
         └───────────────────┴────────────────────┘
                             │
              Evaluation: BERTScore + ROUGE + Latency
              Visualization: 5 charts per run
```

### Design Patterns

| Pattern | Implementation |
|---|---|
| Multi-LLM agents | Each agent uses a different HuggingFace model |
| Shared message bus | All agents share one history |
| Context window | get_recent(n=6) — last 6 messages |
| READ → REASON → WRITE | think_and_act() core loop |
| Round-robin orchestration | run_a2a() deterministic turns |
| Evaluation | BERTScore F1, ROUGE-1, ROUGE-L per response |
| Visualization | 5 charts: BERTScore, Latency, WordCount, ROUGE, Radar |